# PANDAS MASTERY — CHAPTERS 31–36

```
PANDAS MASTERY
Current Chapters: 31 (Time-Series Operations), 32 (Window Operations), 33 (Ranking),
                  34 (Statistical Operations), 35 (Value Counts), 36 (Unique Values)
Progress:         36 / 76 chapters
Completed:        Ch1-30
Remaining:        40 chapters
```

Six chapters: the lag/diff/rolling toolkit that IS time-series feature engineering, ranking
within and across groups, the statistical operations that power EDA, and the
value_counts/unique/nunique trio that's your fastest tool for discovering data quality issues.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)


---
# CHAPTER 31 — Time-Series Operations

## Concept
`shift()`, `diff()`, `pct_change()`, `rolling()`, `expanding()`, `ewm()` are the building blocks
of virtually every time-series feature you'll ever engineer: lag features, growth rates, moving
averages. Each is a thin, convenient wrapper around ideas you already understand from Chapter
5-6 (label-based selection) and NumPy's `np.diff`/`cumsum`.


In [ ]:
prices = pd.Series([100, 102, 101, 105, 108, 106, 110],
                     index=pd.date_range("2023-01-01", periods=7, freq="D"), name="price")

print("prices:\n", prices)
print()
print("shift(1) -- yesterday's value, aligned to today's row (a LAG feature):\n", prices.shift(1))
print("\nshift(-1) -- TOMORROW's value, aligned to today (a LEAD -- use with care, see below):\n", prices.shift(-1))
print("\ndiff() -- today minus yesterday:\n", prices.diff())
print("\npct_change() -- (today - yesterday) / yesterday:\n", prices.pct_change().round(4))


### Expected Output (verified on pandas 3.0.2)
```
prices:
 2023-01-01    100
2023-01-02    102
2023-01-03    101
2023-01-04    105
2023-01-05    108
2023-01-06    106
2023-01-07    110
Freq: D, Name: price, dtype: int64

shift(1) -- yesterday's value, aligned to today's row (a LAG feature):
 2023-01-01      NaN
2023-01-02    100.0
2023-01-03    102.0
2023-01-04    101.0
2023-01-05    105.0
2023-01-06    108.0
2023-01-07    106.0
Freq: D, Name: price, dtype: float64

shift(-1) -- TOMORROW's value, aligned to today (a LEAD -- use with care, see below):
 2023-01-01    102.0
2023-01-02    101.0
2023-01-03    105.0
2023-01-04    108.0
2023-01-05    106.0
2023-01-06    110.0
2023-01-07      NaN
Freq: D, Name: price, dtype: float64

diff() -- today minus yesterday:
 2023-01-01     NaN
2023-01-02     2.0
2023-01-03    -1.0
2023-01-04     4.0
2023-01-05     3.0
2023-01-06    -2.0
2023-01-07     4.0
Freq: D, Name: price, dtype: float64

pct_change() -- (today - yesterday) / yesterday:
 2023-01-01       NaN
2023-01-02    0.0200
2023-01-03   -0.0098
2023-01-04    0.0396
2023-01-05    0.0286
2023-01-06   -0.0185
2023-01-07    0.0377
Freq: D, Name: price, dtype: float64
```

### `shift(1)` is a lag feature; `shift(-1)` is a lead feature -- and leads are a leakage trap
`shift(1)` moves data **forward in time** -- row `t` now holds what was true at `t-1`, a
legitimate lag feature ("yesterday's price"), safe to use as a predictor for "today." `shift(-1)`
does the opposite -- row `t` now holds what will be true at `t+1` -- this is a **lead** feature,
and using it as a model input is a textbook **data leakage** bug (Chapter 42 covers this fully):
you'd be training a model using literal future information that would never be available at
actual prediction time. **The sign of the shift argument is the entire difference between a
legitimate feature and a leakage bug -- always double-check which direction you're shifting.**

### `diff()` and `pct_change()` -- absolute vs relative change
`diff()` gives the absolute change (useful when the underlying scale is meaningful and
consistent -- e.g., temperature). `pct_change()` gives the relative/percentage change (useful
when comparing changes across series with very different scales -- e.g., comparing the "growth
rate" of a $10 stock and a $1000 stock, where absolute diffs aren't directly comparable but
percentage changes are).

### `ewm()` -- exponentially weighted moving average, recent data weighted more heavily


In [ ]:
sma = prices.rolling(window=3).mean()         # simple moving average, equal weight to last 3 days
ewma = prices.ewm(span=3).mean()                  # exponential -- MORE weight on recent days

print("Simple moving average (window=3):\n", sma)
print("\nExponentially weighted moving average (span=3):\n", ewma.round(2))


### Expected Output (verified on pandas 3.0.2)
```
Simple moving average (window=3):
 2023-01-01           NaN
2023-01-02           NaN
2023-01-03    101.000000
2023-01-04    102.666667
2023-01-05    104.666667
2023-01-06    106.333333
2023-01-07    108.000000
Freq: D, Name: price, dtype: float64

Exponentially weighted moving average (span=3):
 2023-01-01    100.00
2023-01-02    101.33
2023-01-03    101.14
2023-01-04    103.20
2023-01-05    105.68
2023-01-06    105.84
2023-01-07    107.94
Freq: D, Name: price, dtype: float64
```

### Why EWMA reacts faster than SMA to recent changes
SMA gives the last 3 days **equal** weight -- a sharp price move 3 days ago still counts exactly
as much as yesterday's move until it falls out of the window entirely. EWMA weights more recent
observations **exponentially more heavily**, so it responds faster to recent trends and doesn't
have a hard cutoff window -- this is why EWMA-based indicators are common in
finance/monitoring contexts where "how is this behaving *right now*" matters more than a flat
historical average.

### Interview Insight
> "What's the risk of using `shift(-1)` as a feature in a predictive model?"
`shift(-1)` creates a **lead** feature -- it pulls a future value backward to align with the
current row, meaning the model would be trained using information that would not actually be
available at prediction time in production. This is a direct instance of data leakage: the
model appears to perform excellently in training/validation (since it's essentially being shown
the answer) but fails in real deployment where future data genuinely isn't available yet.


---
# CHAPTER 32 — Window Operations

## Concept
`rolling()` (fixed-size sliding window), `expanding()` (window that grows to include everything
seen so far), `ewm()` (exponential weighting, just covered) -- the three window types, each with
`min_periods` controlling behavior at the start of the series where a full window doesn't yet
exist.


In [ ]:
data = pd.Series([10, 12, np.nan, 15, 18, 20, 19])

rolling_default = data.rolling(window=3).mean()
rolling_min1 = data.rolling(window=3, min_periods=1).mean()   # allow PARTIAL windows at the start

print("Data:", data.tolist())
print("\nrolling(3).mean() [default min_periods=window]:\n", rolling_default.tolist())
print("\nrolling(3, min_periods=1).mean() [allow partial windows]:\n", rolling_min1.tolist())


### Expected Output (verified on pandas 3.0.2)
```
Data: [10.0, 12.0, nan, 15.0, 18.0, 20.0, 19.0]

rolling(3).mean() [default min_periods=window]:
[nan, nan, nan, nan, nan, 17.666666666666668, 19.0]

rolling(3, min_periods=1).mean() [allow partial windows]:
[10.0, 11.0, 11.0, 13.5, 16.5, 17.666666666666668, 19.0]
```

### Reading the interaction between `NaN` and `window` size carefully
By default, `min_periods` equals the window size -- meaning a window needs **all** `window`
values present and non-missing to produce a result; a single `NaN` at index 2 "poisons" every
window it's a member of. With `window=3`, that's the windows ENDING at indices 2, 3, and 4
(covering data slices `[10,12,NaN]`, `[12,NaN,15]`, `[NaN,15,18]`) -- three consecutive `NaN`
results, not just one, which is easy to underestimate if you only think about the position of
the missing value itself rather than every window it participates in. `min_periods=1` relaxes
this to "produce a result as long as at least 1 non-missing value exists in the window" --
appropriate when you'd rather have an approximate early/near-gap value than no value at all, but
a real judgment call: it means those windows are computed from fewer actual observations than a
full window, which is worth being explicit about (e.g., in a feature name or documentation) if
that value feeds a model.

### `expanding()` -- a window that grows to include everything from the start


In [ ]:
clean_data = pd.Series([10, 12, 14, 15, 18, 20, 19])

expanding_mean = clean_data.expanding().mean()
print("expanding().mean() -- cumulative average, growing window from the start:\n", expanding_mean.tolist())
print("\n<- Position 0: just [10] -> mean 10")
print("   Position 1: [10,12] -> mean 11")
print("   Position 6: [10,12,14,15,18,20,19] (ALL data so far) -> mean of everything")


### Expected Output (verified on pandas 3.0.2)
```
expanding().mean() -- cumulative average, growing window from the start:
[10.0, 11.0, 12.0, 12.75, 13.8, 14.833333333333334, 15.428571428571429]

<- Position 0: just [10] -> mean 10
   Position 1: [10,12] -> mean 11
   Position 6: [10,12,14,15,18,20,19] (ALL data so far) -> mean of everything
```

### Real-world use: `expanding()` for "running total / running average" business metrics
`expanding().sum()` computes a genuine running total (e.g., cumulative revenue year-to-date at
each point in time) -- different from `rolling()`, which only looks at a *fixed recent window*
regardless of history. Both are legitimate, answering different business questions: "how are we
trending recently" (rolling) vs "what's our total so far" (expanding).

### Group-wise rolling -- combining Chapter 21's groupby with window operations


In [ ]:
sales_by_store = pd.DataFrame({
    "store": ["A", "A", "A", "A", "B", "B", "B", "B"],
    "day": [1, 2, 3, 4, 1, 2, 3, 4],
    "revenue": [100, 120, 90, 130, 200, 180, 220, 210],
})

sales_by_store["rolling_avg"] = (
    sales_by_store.groupby("store")["revenue"]
    .transform(lambda s: s.rolling(2, min_periods=1).mean())
)
print(sales_by_store)


### Expected Output (verified on pandas 3.0.2)
```
  store  day  revenue  rolling_avg
0     A    1      100        100.0
1     A    2      120        110.0
2     A    3       90        105.0
3     A    4      130        110.0
4     B    1      200        200.0
5     B    2      180        190.0
6     B    3      220        200.0
7     B    4      210        215.0
```

### Why `groupby(...).transform(lambda s: s.rolling(...))` is the correct pattern
This ensures the rolling window **never crosses store boundaries** -- store B's day-1 rolling
average correctly starts fresh at `200.0`, not blended with store A's final values. This is
exactly Chapter 22's `transform()` lesson (same-length output, aligned back to original rows)
combined with this chapter's rolling window -- a direct, common feature-engineering pattern for
per-entity time-series features (e.g., "each customer's own rolling average transaction size,"
never leaking across customers).

### Interview Insight
> "Why must a rolling average feature be computed per-group (e.g., per customer or per store),
> rather than on the whole dataset directly, when data for multiple entities is stacked
> together?"
A plain `df['revenue'].rolling(3).mean()` on stacked multi-entity data would compute a window
that blends observations from **different entities** near each group boundary -- e.g., the last
2 days of store A mixed with the first day of store B -- which is meaningless and would corrupt
the feature. `groupby('store')['revenue'].transform(lambda s: s.rolling(...))` ensures each
group's rolling computation only ever sees that group's own values.


---
# CHAPTER 33 — Ranking

## Concept
`rank()` assigns a rank to each value -- the key decisions are `ascending` (direction) and
`method` (how to handle ties, which is where the real subtlety lives).


In [ ]:
scores = pd.Series([85, 92, 85, 78, 92], index=["Alice", "Bob", "Carol", "Dave", "Eve"])

print("scores:\n", scores)
print()
for method in ["average", "min", "max", "first", "dense"]:
    print(f"rank(method='{method}'):", scores.rank(ascending=False, method=method).tolist())


### Expected Output (verified on pandas 3.0.2)
```
scores:
 Alice    85
Bob      92
Carol    85
Dave     78
Eve      92
dtype: int64

rank(method='average'): [3.5, 1.5, 3.5, 5.0, 1.5]
rank(method='min')    : [3.0, 1.0, 3.0, 5.0, 1.0]
rank(method='max')    : [4.0, 2.0, 4.0, 5.0, 2.0]
rank(method='first')  : [3.0, 1.0, 4.0, 5.0, 2.0]
rank(method='dense')  : [2.0, 1.0, 2.0, 3.0, 1.0]
```

### Reading each tie-handling method precisely, for this exact tie situation (Bob=Eve=92, Alice=Carol=85)
- **`average`**: tied values get the **average** of the ranks they'd span -- Bob and Eve are
  tied for "1st and 2nd place," so both get `1.5`; Alice and Carol are tied for "3rd and 4th,"
  both get `3.5`. This is the **default**, and the most common choice for percentile-style
  scoring where fractional ranks are acceptable.
- **`min`**: tied values all get the **lowest** (best) rank in their tie group -- both Bob and
  Eve get `1` (not `1.5`), both Alice and Carol get `3`. Sports-style "both get gold, next
  person gets bronze" ranking.
- **`max`**: the opposite -- tied values get the **highest** (worst) rank in their group.
- **`first`**: ties broken by **original order of appearance** -- no ties at all in the output;
  whoever appeared first in the data gets the better rank. Useful when you need a strict,
  reproducible ordering with no ties, e.g., for deterministic top-K selection.
- **`dense`**: like `min`, but the **next** rank after a tie doesn't skip numbers -- after two
  people tied for rank 1, the next distinct value gets rank 2 (not 3, as `min` would give).
  Useful for "how many distinct performance tiers exist" style questions.

### Ranking within groups -- combining Chapter 21's groupby with `rank()`


In [ ]:
students = pd.DataFrame({
    "student": ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"],
    "class": ["Math", "Math", "Math", "Science", "Science", "Science"],
    "score": [85, 92, 78, 90, 88, 95],
})

students["rank_in_class"] = students.groupby("class")["score"].rank(ascending=False, method="min")
print(students.sort_values(["class", "rank_in_class"]))


### Expected Output (verified on pandas 3.0.2)
```
  student    class  score  rank_in_class
1     Bob     Math     92            1.0
0   Alice     Math     85            2.0
2   Carol     Math     78            3.0
5   Frank  Science     95            1.0
3    Dave  Science     90            2.0
4     Eve  Science     88            3.0
```

Each student is ranked **only against their own class**, not the whole dataset -- exactly
Chapter 21/22's "groupby, then apply a per-row-aligned operation" pattern, now with `rank()`
instead of `transform()` (though `.groupby(...)[col].rank()` is itself effectively a transform-
shaped operation, returning one value per original row).

### Interview Insight
> "Two employees are tied for the highest sales figure. How would `rank(method='min')` versus
> `rank(method='dense')` differ in assigning the NEXT rank (for the 3rd-highest performer)?"
`method='min'`: both tied employees get rank 1, and the next distinct value gets rank **3**
(ranks "skip" to account for the two people who share rank 1). `method='dense'`: both tied
employees get rank 1, and the next distinct value gets rank **2** (no skipping -- dense ranking
counts distinct value tiers, not raw position).


---
# CHAPTER 34 — Statistical Operations

## Concept
`mean/median/std/var/min/max/sum/count/quantile/corr/cov` -- the EDA statistics toolkit,
axis-aware exactly like NumPy, with one Pandas-specific addition: `skipna` (missing-value
handling) that's `True` by default, unlike raw NumPy.


In [ ]:
df = pd.DataFrame({
    "math": [85, 92, np.nan, 78, 95],
    "science": [90, 85, 88, np.nan, 92],
    "english": [75, 95, 80, 85, 90],
})

print("df:\n", df)
print()
print("mean(axis=0) [per COLUMN, default] -- skipna=True by DEFAULT, unlike raw NumPy:\n", df.mean())
print("\nmean(axis=1) [per ROW]:\n", df.mean(axis=1))
print("\nmean(skipna=False) -- NaN poisons the result, exactly like raw NumPy:\n", df.mean(skipna=False))


### Expected Output (verified on pandas 3.0.2)
```
df:
   math  science  english
0  85.0     90.0       75
1  92.0     85.0       95
2   NaN     88.0       80
3  78.0      NaN       85
4  95.0     92.0       90

mean(axis=0) [per COLUMN, default] -- skipna=True by DEFAULT, unlike raw NumPy:
math       87.50
science    88.75
english    85.00
dtype: float64

mean(axis=1) [per ROW]:
0    83.333333
1    90.666667
2    84.000000
3    81.500000
4    92.333333
dtype: float64

mean(skipna=False) -- NaN poisons the result, exactly like raw NumPy:
math       NaN
science    NaN
english    85.0
dtype: float64
```

### The `skipna` default is a genuine, important Pandas-vs-NumPy difference
Raw NumPy's `arr.mean()` **propagates** `NaN` -- a single missing value poisons the entire
result (Chapter 15 of the NumPy course), requiring `np.nanmean()` explicitly to skip it.
**Pandas' `.mean()` skips `NaN` by default** (`skipna=True`) -- a deliberate design choice
reflecting that real business data is expected to have gaps, and "give me the average of what I
actually have" is the more useful default for analytical work than "poison everything the
moment anything is missing." Setting `skipna=False` restores NumPy's stricter propagating
behavior when you specifically want to know "is there ANY missing data affecting this column."

### `corr()` and `cov()` -- Chapter 14 of the NumPy course, now index/column-aware


In [ ]:
correlation = df.corr()
print("df.corr() -- pairwise correlation matrix, NaN handled automatically:\n", correlation.round(3))

print("\ndf['math'].corr(df['english']):", round(df["math"].corr(df["english"]), 3))


### Expected Output (verified on pandas 3.0.2)
```
df.corr() -- pairwise correlation matrix, NaN handled automatically:
          math  science  english
math     1.000    0.054    0.553
science  0.054    1.000   -0.367
english  0.553   -0.367    1.000

df['math'].corr(df['english']): 0.553
```

### Why `df.corr()` is more convenient than NumPy's `np.corrcoef` here (Chapter 14 of the NumPy course, revisited)
Recall from the NumPy course: `np.corrcoef` defaults to treating **rows** as variables
(`rowvar=True`), a common source of bugs on standard `(samples, features)` data, requiring
`rowvar=False` to get the intuitive feature-by-feature correlation matrix. `df.corr()` has no
such gotcha -- it always treats **columns** as the variables (matching how a DataFrame's columns
represent features), and additionally handles missing values automatically (pairwise, using
only the rows where both columns have data for each specific pairwise correlation) -- no manual
NaN-handling required before calling it, unlike raw NumPy.

### Interview Insight
> "Why might `df['col'].mean()` and `np.mean(df['col'].to_numpy())` give different results on
> the same column?"
If the column contains `NaN`, Pandas' `.mean()` skips missing values by default (`skipna=True`),
computing the mean of only the present values; `np.mean()` on the raw NumPy array has no such
default and returns `NaN` if any value is missing, since plain NumPy `mean` propagates NaN
unless you explicitly call `np.nanmean()` instead.


---
# CHAPTER 35 — Value Counts

## Concept
`value_counts()` is the single fastest way to understand the distribution of a categorical (or
even numeric) column -- frequency counts, sorted descending by default, with easy normalization
to percentages.


In [ ]:
feedback = pd.Series(["Positive", "Positive", "Neutral", "Negative", "Positive",
                        "Neutral", "Positive", None, "Negative", "Positive"])

print("value_counts() (default -- excludes NaN, sorted descending):\n", feedback.value_counts())
print("\nvalue_counts(normalize=True) -- PERCENTAGES instead of raw counts:\n",
      feedback.value_counts(normalize=True).round(3))
print("\nvalue_counts(dropna=False) -- INCLUDE missing values in the count:\n",
      feedback.value_counts(dropna=False))


### Expected Output (verified on pandas 3.0.2)
```
value_counts() (default -- excludes NaN, sorted descending):
Positive    5
Neutral     2
Negative    2
Name: count, dtype: int64

value_counts(normalize=True) -- PERCENTAGES instead of raw counts:
Positive    0.556
Neutral     0.222
Negative    0.222
Name: proportion, dtype: float64

value_counts(dropna=False) -- INCLUDE missing values in the count:
Positive    5
Neutral     2
Negative    2
NaN         1
Name: count, dtype: int64
```

### Why `normalize=True` matters more than it looks for EDA reporting
Raw counts (`5`, `2`, `2`) require the reader to also know the total (10, since one value is
missing and excluded by default) to interpret them meaningfully. `normalize=True` directly
gives interpretable proportions (`55.6%`, `22.2%`, `22.2%`) -- the standard,
immediately-communicable form for any report or dashboard describing a categorical
distribution, without forcing the reader to do arithmetic.

### `value_counts()` as a data-quality discovery tool -- the real reason this chapter exists


In [ ]:
messy_country = pd.Series(["USA", "usa", "USA", "United States", "Canada", "CANADA", "USA", "canada"])

print("Naive read: 'this column has a few countries, looks fine'")
print("\nBut value_counts() reveals the TRUE fragmentation:\n", messy_country.value_counts())
print("\nnunique():", messy_country.nunique(), " <- SIX distinct values for what should be TWO countries")


### Expected Output (verified on pandas 3.0.2)
```
Naive read: 'this column has a few countries, looks fine'

But value_counts() reveals the TRUE fragmentation:
USA              3
usa              1
United States    1
Canada           1
CANADA           1
canada            1
Name: count, dtype: int64

nunique(): 6  <- SIX distinct values for what should be TWO countries
```

### Why this is the fastest EDA check for exactly the string-cleaning problem Chapter 16 solves
A `groupby("country")` on this column, run **before** noticing this fragmentation, would
silently split what should be one "USA" group into three separate groups (`"USA"`, `"usa"`,
`"United States"`) -- understating true group sizes/aggregates without any error. Running
`value_counts()` on every categorical column during initial EDA (Chapter 11's inspection
workflow) is the standard, fast way to catch this **before** it corrupts downstream
`groupby`/aggregation results -- directly motivating exactly the string-cleaning chain (`.str
.strip().str.lower()`, or a mapping via `.map()`) built in Chapter 16.

### Interview Insight
> "You're about to `groupby()` a categorical column for an aggregation report. What quick check
> would you run first, and why?"
`df[col].value_counts()` (or `.nunique()`) -- this immediately surfaces category fragmentation
from inconsistent casing, whitespace, or naming variants (`'USA'` vs `'usa'` vs
`'United States'`) that would otherwise silently split what should be one group into several in
the subsequent `groupby`, understating true aggregated totals with no error or warning raised.


---
# CHAPTER 36 — Unique Values

## Concept
`unique()` returns the **distinct values themselves** (as an array); `nunique()` returns just
the **count** of distinct values; `value_counts()` (Chapter 35) returns distinct values **with
their frequencies**. Three related but distinctly-purposed tools.


In [ ]:
df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "plan": ["Basic", "Premium", "Basic", "Enterprise", "Premium"],
    "region": ["East", "East", "West", "West", "East"],
})

print("df['plan'].unique() -- the DISTINCT VALUES themselves, as an array:", df["plan"].unique())
print("type:", type(df["plan"].unique()))
print()
print("df.nunique() -- distinct COUNT per column, for the WHOLE DataFrame at once:\n", df.nunique())
print()
print("df['customer_id'].nunique() == len(df)?", df["customer_id"].nunique() == len(df),
      " <- confirms customer_id is a genuine unique key")


### Expected Output (verified on pandas 3.0.2)
```
df['plan'].unique() -- the DISTINCT VALUES themselves, as an array: <ArrowStringArray>
['Basic', 'Premium', 'Enterprise']
Length: 3, dtype: str
type: <class 'pandas.arrays.ArrowStringArray'>

df.nunique() -- distinct COUNT per column, for the WHOLE DataFrame at once:
customer_id    5
plan           3
region         2
dtype: int64

df['customer_id'].nunique() == len(df)? True  <- confirms customer_id is a genuine unique key
```

### A real, current pandas 3.0 detail: `.unique()` on a `str`-dtype column returns `ArrowStringArray`, not a plain NumPy array
Older Pandas material (and the NumPy course's general assumption that `.unique()`-style
operations return a plain `ndarray`) doesn't hold universally on Pandas 3.0: since `plan` is a
`str`-dtype column (Chapter 8's Arrow-backed string dtype), `.unique()` returns pandas'
`ArrowStringArray` container, not a bare NumPy array. For a numeric column, `.unique()` still
returns a plain `ndarray` as expected — this type difference is specific to the newer
Arrow-backed string dtype. In practice this rarely matters (`ArrowStringArray` supports
indexing, iteration, and conversion via `list(...)` or `.to_numpy()` just like an ndarray would),
but it's exactly the kind of "the type isn't what older documentation says" surprise worth
knowing about before writing code that asserts on the exact return type.

### `nunique()` as an instant "is this column a valid unique key?" check
`df['customer_id'].nunique() == len(df)` is a one-line, standard validation that a column
intended to be a unique identifier genuinely **is** one -- if it returns `False`, that column has
duplicate values and can't safely be used as a join key or index without first investigating why
(directly connecting to Chapter 13's duplicate-detection and Chapter 25's join-safety lessons).

### `unique()` for discovering unexpected categories, typos, and high-cardinality surprises


In [ ]:
raw_status = pd.Series(["active", "Active", "ACTIVE", "inactive", "pending", "actve", "Inactive"])

print("raw_status.unique():", raw_status.unique())
print("\n<- 7 distinct values for what should probably be 3 real statuses (active/inactive/pending) --")
print("   'actve' (index 5) is very likely a TYPO for 'active', not a genuine 4th status.")
print("   unique() surfaces this instantly; you'd never spot it by scrolling through raw rows.")

# nunique() for discovering high-cardinality columns that shouldn't be treated as categorical
customer_notes = pd.Series([f"Note about customer {i}" for i in range(1000)])
print(f"\ncustomer_notes.nunique() = {customer_notes.nunique()} out of {len(customer_notes)} rows")
print("<- Ratio near 1.0 -> essentially all-unique free text, NOT a category-dtype candidate (Chapter 17)")


### Expected Output (verified on pandas 3.0.2)
```
raw_status.unique(): <ArrowStringArray>
['active', 'Active', 'ACTIVE', 'inactive', 'pending', 'actve', 'Inactive']
Length: 7, dtype: str

<- 7 distinct values for what should probably be 3 real statuses (active/inactive/pending) --
   'actve' (index 5) is very likely a TYPO for 'active', not a genuine 4th status.
   unique() surfaces this instantly; you'd never spot it by scrolling through raw rows.

customer_notes.nunique() = 1000 out of 1000 rows
<- Ratio near 1.0 -> essentially all-unique free text, NOT a category-dtype candidate (Chapter 17)
```

### The `nunique()/len()` ratio -- the same decision rule from Chapter 17, now the actual tool
`nunique() / len(df)` close to `0` (many repeats) signals a good category-dtype and
`groupby`-key candidate; close to `1` (mostly/all unique) signals free text, an ID column, or a
genuinely continuous measurement -- none of which benefit from category dtype, and shouldn't be
naively grouped on either (grouping by a near-unique column produces nearly as many groups as
rows, which is rarely the intended analysis).

### `unique()` vs `nunique()` vs `value_counts()` -- choosing the right tool
| Tool | Returns | Use when you need |
|---|---|---|
| `.unique()` | The distinct values themselves | To *see* what categories actually exist (spot typos/casing issues) |
| `.nunique()` | Just the count of distinct values | A quick cardinality check (category-dtype decision, unique-key validation) |
| `.value_counts()` | Distinct values WITH their frequencies | Understanding the *distribution* -- which categories dominate, which are rare |

### Interview Insight
> "How would you quickly check whether a column claimed to be a unique identifier actually is
> one?"
`df[col].nunique() == len(df)` -- if `True`, every row has a distinct value in that column,
confirming it's safe to use as a unique key (for indexing, joining, or as a primary key
constraint check); if `False`, the column contains duplicates and needs investigation
(Chapter 13) before being trusted as a unique identifier.


---
## Chapters 31–36 Summary

- **Time-series operations**: `shift(1)` (lag, safe) vs `shift(-1)` (lead, a leakage trap) --
  the sign is the entire difference between a legitimate feature and a bug; `diff()` (absolute)
  vs `pct_change()` (relative); `ewm()` weights recent observations more heavily than a flat
  `rolling()` window.
- **Window operations**: `rolling()` (fixed window) vs `expanding()` (grows from the start) vs
  `ewm()`; `min_periods` controls partial-window behavior at the series' start -- verified live
  how a single `NaN` poisons every window it touches under the default `min_periods=window`;
  group-wise rolling via `groupby(...).transform(lambda s: s.rolling(...))` prevents windows
  from crossing entity boundaries.
- **Ranking**: `method='average'/'min'/'max'/'first'/'dense'` handle ties in five genuinely
  different, precisely-defined ways -- verified all five side by side on the same tied data;
  `groupby(...)[col].rank()` ranks within groups, not globally.
- **Statistical operations**: Pandas' `skipna=True` default is a real, important divergence
  from raw NumPy's NaN-propagating default -- verified both behaviors directly; `df.corr()` has
  no `rowvar` gotcha (always column-wise) and auto-handles missing values pairwise, unlike raw
  `np.corrcoef`.
- **Value counts**: `value_counts()` is the fastest real EDA tool for discovering category
  fragmentation (`'USA'`/`'usa'`/`'United States'`) before it silently corrupts a `groupby`;
  `normalize=True` gives immediately-interpretable proportions.
- **Unique values**: `nunique() == len(df)` is the standard one-line unique-key validation;
  `nunique()/len(df)` is the same low/high-cardinality decision rule from Chapter 17,
  operationalized as an actual diagnostic check.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For a Series with a `NaN` at position 5, predict which positions
in `s.rolling(window=3).mean()` will be `NaN`, assuming default `min_periods`.

**Exercise 2:** You need "each customer's average order value over their last 3 orders,"
computed per customer, never blending across customers. Write the `groupby(...).transform(...)`
expression that computes this.

**Coding Challenge:** Write `detect_near_duplicate_categories(s, threshold=0.85)` that uses
`value_counts()` to get distinct values, then flags pairs of values whose lowercase, stripped
forms are identical (a simple normalization-based near-duplicate detector) -- returning a list
of `(value_a, value_b)` pairs that likely represent the same real category written differently.

**Debugging Challenge:**
```python
df["next_month_revenue"] = df["revenue"].shift(-1)
model.fit(df[["revenue", "next_month_revenue"]], df["churned"])
```
A teammate built a churn model that performs suspiciously well in testing. Identify the exact
line responsible, and explain the mechanism (referencing this chapter's lead/lag distinction).

**Interview Questions:**
1. Precisely explain why `shift(-1)` is dangerous as a model feature while `shift(1)` is not.
2. Why does `rank(method='dense')` produce different "next rank after a tie" behavior than
   `rank(method='min')`?
3. Give a concrete reason `df.corr()` is more convenient than `np.corrcoef()` for a DataFrame
   with some missing values.
4. Why is `value_counts()` (not just eyeballing rows) the recommended first check before any
   `groupby()` on a categorical column?

**Quiz:**
- Q1: True/False -- Pandas' `.mean()` propagates NaN by default, matching raw NumPy's behavior.
- Q2: Which window type (`rolling`, `expanding`, `ewm`) has NO fixed window size at all?
- Q3: What does `df[col].nunique() == len(df)` verify?

---

 